In [2]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

plt.style.use('dark_background')



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import plotly.io as pio

pio.renderers.default = 'iframe'

/kaggle/input/datasets/krupalpatel07/mercedes-benz-group/Mercedes-Benz Group.csv


In [3]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/mercedes-benz-group/Mercedes-Benz Group.csv"
df = pd.read_csv(file_path)

In [4]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [5]:
# =====================================================
# 4. LUXURY HEADER
# =====================================================
from IPython.display import display, HTML

def luxury_black(title):
    display(HTML(f"""
    <div style="
        background: linear-gradient(90deg, #000000, #434343);
        padding: 20px; border-radius: 12px; margin-top:20px;">
        <h1 style="color:#e5e5e5; text-align:center;">{title}</h1>
    </div>
    """))

luxury_black("📊 Price Evolution")

In [6]:
# =====================================================
# 5. PRICE VISUAL
# =====================================================
fig = px.line(df, y='close', title='Mercedes-Benz Price')
fig.show()


In [7]:
# =====================================================
# 6. FEATURE ENGINEERING
# =====================================================
luxury_black("🧠 Feature Engineering")

df['returns'] = df['close'].pct_change()
df['ma_20'] = df['close'].rolling(20).mean()
df['vol'] = df['returns'].rolling(20).std()

df = df.dropna()

In [8]:
# =====================================================
# 7. DATA SCALING
# =====================================================
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[['close']])

# Create sequences
X, y = [], []
window = 20
for i in range(window, len(scaled)):
    X.append(scaled[i-window:i])
    y.append(scaled[i])

X, y = np.array(X), np.array(y)

# Split
split = int(len(X)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [9]:
# =====================================================
# 8. LSTM MODEL
# =====================================================
luxury_black("🤖 Deep Learning Model (LSTM)")

model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1],1)))
model.add(LSTM(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)

# Predictions
preds = model.predict(X_test)

# Inverse scaling
preds = scaler.inverse_transform(preds)
y_test_real = scaler.inverse_transform(y_test)

rmse = np.sqrt(mean_squared_error(y_test_real, preds))
print("RMSE:", rmse)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test_real.flatten(), name='Actual'))
fig.add_trace(go.Scatter(y=preds.flatten(), name='Predicted'))
fig.show()

2026-05-03 12:30:33.758884: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Epoch 1/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 0.0058
Epoch 2/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 2.1322e-04
Epoch 3/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 2.0190e-04
Epoch 4/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 1.6554e-04
Epoch 5/5
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 1.7350e-04
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
RMSE: 1.9559616437769136


In [10]:
# =====================================================
# 9. ATTENTION INTUITION (WEIGHTED RECENT DATA)
# =====================================================
luxury_black("⚡ Attention Intuition")

weights = np.linspace(1, 2, window)
weights = weights / weights.sum()

attn_signal = []
for i in range(window, len(df)):
    segment = df['close'].values[i-window:i]
    attn_signal.append(np.sum(segment * weights))

attn_series = pd.Series(attn_signal, index=df.index[window:])

fig = px.line(attn_series, title='Attention Weighted Signal')
fig.show()

In [11]:
# =====================================================
# 10. HYBRID SIGNAL
# =====================================================
luxury_black("🎯 Hybrid Alpha Engine")

aligned_attn = attn_series.reindex(df.index).fillna(method='bfill')

signal = (df['close'] > aligned_attn) & (df['vol'] < df['vol'].rolling(50).mean())
df['signal'] = signal.astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Price'))
fig.add_trace(go.Scatter(x=df.index[df['signal']==1], y=df['close'][df['signal']==1],
                         mode='markers', name='Buy'))
fig.show()


/tmp/ipykernel_57/240099774.py:6: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [12]:
# =====================================================
# 11. FINAL INSIGHTS
# =====================================================
luxury_black("📌 Luxury Quant Insights")

print("""
1. LSTM captures sequential dependencies.
2. Attention weighting emphasizes recent importance.
3. Hybrid signals combine ML + statistical logic.
4. Volatility filtering improves stability.
5. Deep learning enhances nonlinear pattern capture.
""")

# =====================================================
# END
# =====================================================



1. LSTM captures sequential dependencies.
2. Attention weighting emphasizes recent importance.
3. Hybrid signals combine ML + statistical logic.
4. Volatility filtering improves stability.
5. Deep learning enhances nonlinear pattern capture.

